# 02 · Preparação dos Dados de Treinamento

Gera pares `(query, documento_relevante)` a partir dos qrels para o fine-tuning.

**Estratégia:** pares positivos para `MultipleNegativesRankingLoss`  
*(os demais exemplos no batch atuam como negativos implícitos)*

## 1 · Imports e configuração

In [1]:
import json, random
from collections import defaultdict
from pathlib import Path
from tqdm.notebook import tqdm

DATA_DIR      = Path('data')
MIN_RELEVANCE = 1     # relevância mínima para considerar positivo
MAX_POSITIVES = 3     # máx. de docs positivos por query
NEG_RATIO     = 1     # negativos por positivo (0 = só pares positivos)
RANDOM_SEED   = 42
VAL_RATIO     = 0.1  # 5% para validação

random.seed(RANDOM_SEED)

## 2 · Funções de carregamento

In [2]:
def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f]

def load_qrels(path):
    qrels = defaultdict(dict)
    with open(path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 4:
                qid, _, did, rel = parts
                qrels[qid][did] = int(rel)
    return qrels

## 3 · Carregamento dos dados

In [3]:
queries = load_jsonl(DATA_DIR / 'queries.jsonl')
corpus  = load_jsonl(DATA_DIR / 'corpus.jsonl')
qrels   = load_qrels(DATA_DIR / 'qrels.txt')

query_map   = {q['query_id']: q['text'] for q in queries}
doc_map     = {d['doc_id']: d['text']   for d in corpus}
all_doc_ids = list(doc_map.keys())

print(f'Queries        : {len(queries):,}')
print(f'Documentos     : {len(corpus):,}')
print(f'Queries c/qrels: {len(qrels):,}')

Queries        : 1,444
Documentos     : 369,721
Queries c/qrels: 1,444


## 4 · Construção dos pares de treinamento

In [4]:
pairs, skipped = [], 0

for qid, rels in tqdm(qrels.items(), desc='Construindo pares'):
    if qid not in query_map:
        skipped += 1
        continue

    query_text = query_map[qid]
    pos_ids = [did for did, rel in rels.items()
               if rel >= MIN_RELEVANCE and did in doc_map]
    rel_set = set(rels.keys())

    if not pos_ids:
        skipped += 1
        continue

    random.shuffle(pos_ids)
    pos_ids = pos_ids[:MAX_POSITIVES]

    for pos_id in pos_ids:
        if NEG_RATIO > 0:
            neg_candidates = [d for d in all_doc_ids if d not in rel_set]
            neg_ids = random.sample(neg_candidates, min(NEG_RATIO, len(neg_candidates)))
            for neg_id in neg_ids:
                pairs.append({
                    'query': query_text,
                    'positive': doc_map[pos_id],
                    'negative': doc_map[neg_id],
                })
        else:
            pairs.append({'query': query_text, 'positive': doc_map[pos_id]})

print(f'Pares gerados       : {len(pairs):,}')
print(f'Queries sem match   : {skipped}')

Construindo pares:   0%|          | 0/1444 [00:00<?, ?it/s]

Pares gerados       : 4,332
Queries sem match   : 0


## 5 · Divisão treino / validação

In [11]:
random.shuffle(pairs)
n_val       = max(1, int(len(pairs) * VAL_RATIO))
train_pairs = pairs[n_val:]
val_pairs   = pairs[:n_val]

print(f'Treino : {len(train_pairs):,}')
print(f'Val    : {len(val_pairs):,}')

Treino : 3,899
Val    : 433


## 6 · Salvar arquivos

In [12]:
for path, data in [
    (DATA_DIR / 'train_pairs.jsonl', train_pairs),
    (DATA_DIR / 'val_pairs.jsonl',   val_pairs),
]:
    with open(path, 'w', encoding='utf-8') as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    print(f'Salvo → {path}')

Salvo → data\train_pairs.jsonl
Salvo → data\val_pairs.jsonl


## 7 · Amostra de par de treinamento

## 8 · Hard Negative Mining

Usa o índice FAISS do baseline para encontrar documentos que o modelo *quase* acerta — documentos semanticamente próximos da query mas não relevantes. Esses são negativos muito mais difíceis que os aleatórios.

In [13]:
import faiss, pickle, torch
from sentence_transformers import SentenceTransformer

BASELINE_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
INDEX_PATH     = Path('models/baseline/faiss.index')
DOCIDS_PATH    = Path('models/baseline/doc_ids.pkl')
TOP_K_MINE     = 50   # candidatos recuperados por query para minerar negativos
HARD_NEG_PER_Q = 3    # máx. hard negatives por query

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Carregando modelo e índice FAISS...')
mine_model = SentenceTransformer(BASELINE_MODEL, device=device)
mine_index = faiss.read_index(str(INDEX_PATH))
mine_doc_ids = pickle.load(open(DOCIDS_PATH, 'rb'))
print(f'Índice carregado: {mine_index.ntotal:,} vetores')

Carregando modelo e índice FAISS...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Índice carregado: 369,721 vetores


In [14]:
hard_triplets = []

all_query_texts = [query_map[qid] for qid in qrels if qid in query_map]
all_qids        = [qid for qid in qrels if qid in query_map]

# encoda todas as queries de uma vez
q_embs = mine_model.encode(
    all_query_texts, batch_size=256, normalize_embeddings=True,
    convert_to_numpy=True, show_progress_bar=True,
).astype('float32')

_, all_idxs = mine_index.search(q_embs, TOP_K_MINE)

for i, qid in enumerate(tqdm(all_qids, desc='Minerando hard negatives')):
    query_text = query_map[qid]
    rel_set    = set(qrels[qid].keys())
    pos_ids    = [did for did, rel in qrels[qid].items()
                  if rel >= MIN_RELEVANCE and did in doc_map]
    if not pos_ids:
        continue

    # hard negatives: recuperados pelo modelo mas não relevantes
    hard_negs = [
        mine_doc_ids[idx]
        for idx in all_idxs[i]
        if mine_doc_ids[idx] not in rel_set and mine_doc_ids[idx] in doc_map
    ][:HARD_NEG_PER_Q]

    if not hard_negs:
        continue

    random.shuffle(pos_ids)
    for j, pos_id in enumerate(pos_ids[:MAX_POSITIVES]):
        neg_id = hard_negs[j % len(hard_negs)]
        hard_triplets.append({
            'query':    query_text,
            'positive': doc_map[pos_id],
            'negative': doc_map[neg_id],
        })

print(f'Triplets com hard negatives: {len(hard_triplets):,}')

Batches:   0%|          | 0/6 [00:00<?, ?it/s]

Minerando hard negatives:   0%|          | 0/1444 [00:00<?, ?it/s]

Triplets com hard negatives: 4,332


In [15]:
random.shuffle(hard_triplets)
n_val      = max(1, int(len(hard_triplets) * VAL_RATIO))
train_hard = hard_triplets[n_val:]
val_hard   = hard_triplets[:n_val]

for path, data in [
    (DATA_DIR / 'train_hard_triplets.jsonl', train_hard),
    (DATA_DIR / 'val_hard_triplets.jsonl',   val_hard),
]:
    with open(path, 'w', encoding='utf-8') as f:
        for item in data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    print(f'Salvo → {path}  ({len(data):,} exemplos)')

sample = train_hard[0]
print(f'\nAmostra:')
print(f'  Query   : {sample["query"][:80]}')
print(f'  Positivo: {sample["positive"][:80]}...')
print(f'  Negativo: {sample["negative"][:80]}...')

Salvo → data\train_hard_triplets.jsonl  (3,899 exemplos)
Salvo → data\val_hard_triplets.jsonl  (433 exemplos)

Amostra:
  Query   : irish civil war
  Positivo: originally known as the irish women s citizens and local government association ...
  Negativo: it pitted forces led by brian boru high king of ireland against a norse irish al...


In [25]:
sample = train_pairs[110]
print('── Amostra ─────────────────────────────────')
print(f'Query   : {sample["query"][:82]}')
print(f'Positivo: {sample["positive"][:82]}...')
if 'negative' in sample:
    print(f'Negativo: {sample["negative"][:82]}...')

── Amostra ─────────────────────────────────
Query   : szlachta
Positivo: among others he participated at the battle of chocim rawne battle of vienna and pa...
Negativo: it fires unguided and spin stabilized 9m21 rockets frog is a backronym for free ro...
